# GRIDS permission plugin: live demo on Colab
Starts the chat demo from `plugin/`: an assistant (an Ollama model on the GPU) works on AgentDojo's **simulated** mailbox,
calendar and drive, and every action it proposes passes the **Wall** and then **laya-cybersec** (`TextCortex/laya-cybersec`,
322M parameters) in the Gatekeeper slot. The page shows each decision and the compute each part used.

**Before you start:** Runtime → Change runtime type → a GPU (T4 is enough). Run the cells in order. The last setup cell prints
a public `gradio.live` link that anyone can open while this session runs; nothing real is read or sent.

## 1. Settings

In [ ]:
REPO_URL = "https://github.com/AtharvaThorat/permission-that-learn.git"
BRANCH = "11_Nov_Permission_Plugin"       # main, once the plugin is merged
MODEL = "qwen2.5:7b"                      # Ollama model that drives the assistant; must support tool calling
LAYA_DEVICE = "cpu"                       # laya-cybersec on the CPU, as the project intends; "cuda" is faster

WORK, ENV, CACHE = "/content/permission-that-learn", "/content/plugin-env", "/content/hf-cache"
PY = f"{ENV}/bin/python"

## 2. Helpers
Run once; no changes needed.

In [ ]:
import os, subprocess, time, urllib.request


def sh(cmd, check=True, cwd=None):
    # Run a shell command, stream its output, stop on failure.
    process = subprocess.Popen(cmd, shell=True, cwd=cwd or (WORK if os.path.isdir(WORK) else "/content"),
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end="")
    process.wait()
    if check and process.returncode:
        raise RuntimeError(f"Command failed ({process.returncode}): {cmd}")

## 3. Code and Python environment
A separate environment with the repository's Laya pins plus Gradio and MCP (about 3 minutes).

In [ ]:
if not os.path.isdir(WORK):
    sh(f"git clone --branch {BRANCH} {REPO_URL} {WORK}", cwd="/content")
sh(f"git fetch origin {BRANCH} && git checkout {BRANCH} && git pull --ff-only origin {BRANCH}")
print("Commit:", subprocess.check_output(["git", "-C", WORK, "rev-parse", "HEAD"], text=True).strip())
sh("pip install -q uv")
if not os.path.exists(PY):
    sh(f"uv venv {ENV} --python 3.12 --seed -q")
sh(f"uv pip install --python {PY} -q -r environments/laya.txt -r plugin/requirements.txt")

From https://github.com/AtharvaThorat/permission-that-learn
 * branch            11_Nov_Permission_Plugin -> FETCH_HEAD
Already on '11_Nov_Permission_Plugin'
Your branch is up to date with 'origin/11_Nov_Permission_Plugin'.
From https://github.com/AtharvaThorat/permission-that-learn
 * branch            11_Nov_Permission_Plugin -> FETCH_HEAD
Already up to date.
Commit: 0feae40b3cf934dabc099b355e0c299c6a1fa3b4


## 4. Start Ollama and download the assistant model

In [ ]:
if subprocess.run("which ollama", shell=True, capture_output=True).returncode:
    sh("apt-get update -qq && apt-get install -y -qq zstd pciutils lshw")
    sh("curl -fsSL https://ollama.com/install.sh | sh")
ollama = subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "a"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=2)
        break
    except OSError:
        time.sleep(1)
sh(f"ollama pull {MODEL} 2>&1 | tail -n 2")

removing unused layers 
success 


## 5. Launch the demo
Downloads laya-cybersec once (about 680 MB, pinned revision), loads it, and starts the page. Open the printed link.
Each scenario in the page explains what it shows; **Start / reset Workspace** applies a new scenario or protection level.

In [ ]:
log = open("/content/demo.log", "w")
demo = subprocess.Popen([PY, "-u", "-m", "grids_plugin.app", "--model", MODEL, "--laya-cache", CACHE,
                         "--laya-device", LAYA_DEVICE, "--share"], cwd=f"{WORK}/plugin", stdout=log, stderr=subprocess.STDOUT,
                        env={**os.environ, "GRADIO_ANALYTICS_ENABLED": "False", "PYTHONUNBUFFERED": "1"})
url = None
for _ in range(300):
    text = open("/content/demo.log").read()
    url = next((word for word in text.split() if "gradio.live" in word), None)
    if url or demo.poll() is not None:
        break
    time.sleep(2)
print(f"Open the demo: {url}" if url else "No link yet; the end of the log:\n" + text[-3000:])

Open the demo: https://1ae4f1c2694e05bcc6.gradio.live


## 6. Stop

In [ ]:
demo.terminate()
ollama.terminate()
print("Stopped.")

## 7. If something failed
The end of the demo and Ollama logs.

In [ ]:
print(open("/content/demo.log").read()[-4000:])
print(open("/content/ollama.log").read()[-2000:])